In [1]:
!pip -q install chromadb

In [4]:
import chromadb
import pandas as pd
from tqdm.notebook import tqdm
from utils import get_examples_from_df

In [41]:
chroma_client = chromadb.Client()

# switch `create_collection` to `get_or_create_collection` to avoid creating a new collection every time
collection = chroma_client.get_or_create_collection(
    name="requirements",
    metadata={"hnsw:space": "cosine"} # cosine, l2
)

In [12]:
df = pd.read_excel("data/requirements.xlsx")

In [7]:
indexes_to_drop, examples = get_examples_from_df(df, 5)

In [42]:
documents = []
metadatas = []

for e1 in examples.values():
    for e2 in e1:
        documents.append(e2[0])
        metadatas.append({"vector": e2[1]})

In [43]:
# switch `add` to `upsert` to avoid adding the same documents every time
collection.upsert(
    ids=[f"id_{i}"for i in range(len(documents))],
    documents=documents,
    metadatas=metadatas
)

In [15]:
df_reduced = df.drop(index=indexes_to_drop)

In [44]:
t = df_reduced.sample(1)

results = collection.query(
    query_texts = [t.values[0, 0]],
    n_results = 5
)

In [45]:
t.values[0,0],t.values[0,1:]

('ABS and traction control systems must be fully integrated with the wheel speed sensors, ensuring that sensor data is used effectively for vehicle stability interventions',
 array([0, 0, 1, 0, 0], dtype=object))

In [46]:
results["documents"]

[['The traction control system shall respond correctly to wheel speed sensor inputs under all driving conditions to prevent wheel slip',
  'The ABS and ESC systems must detect a wheel speed sensor failure and activate a fail-safe mode, maintaining as much functionality as possible',
  'The system must rapidly process wheel speed information to activate anti-lock braking systems (ABS) and prevent wheel lockup during emergency braking scenarios',
  'The vehicle control system shall counteract any vehicle handling issues due to wheel speed sensor faults, within system limits',
  'The vehicle control system must provide feedback for brake modulation through the acceleration pedal to assist with precise wheel steering angle input']]

In [47]:
results["metadatas"]

[[{'vector': '[0,0,1,0,0]'},
  {'vector': '[0,0,1,0,0]'},
  {'vector': '[0,0,1,0,0]'},
  {'vector': '[0,0,1,0,0]'},
  {'vector': '[1,1,0,0,0]'}]]